# Lab 05 · A skill for a short architecture decision record

**Prepared by Mustafa Assaf** · 15–20 minute lab

## Package a procedure as a reusable skill

**So far:** each lab wrote its instructions directly into the prompt.

**New:** *skills*. A skill is a `SKILL.md` file: a short header with a `name` and a `description`, then the instructions. An agent first sees only the names and descriptions of all its skills. It picks one, and only then loads that skill's full instructions. This is called *progressive disclosure*, and it keeps the prompt small even when there are many skills.

Our main skill writes a short *architecture decision record* (ADR) with three lines: Context, Decision, and Consequence.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, save `FREEINFERENCE_API_KEY` in Colab Secrets and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/aebb52597ab982f0b8b2f0832815b73dd6aca416.zip"
from coursekit import make_ask
MODE = "practice"
ask = make_ask(mode=MODE)

## Step 1 · Read a skill file

The skill ships inside the `coursekit` package. The header sits between two `---` lines. `read_skill` splits the file there, turns each `key: value` header line into a dictionary entry, and keeps the rest as the instruction body.

In [ ]:
from importlib.resources import files

def read_skill(text):
    """Split a SKILL.md file into its header fields and its instruction body."""
    _, header, body = text.split("---", 2)
    fields = dict(line.split(": ", 1) for line in header.strip().splitlines())
    return fields, body.strip()

adr_skill = files("coursekit").joinpath("skills/architecture-decision/SKILL.md").read_text()
fields, body = read_skill(adr_skill)
print(fields)
print(body)

## Step 2 · Build a catalog of skills

A real agent has many skills. We add two more, written as `SKILL.md` text in the notebook. The *catalog* lists only each skill's name and description. That is all the model sees when it chooses.

In [ ]:
incident_skill = """---
name: incident-summary
description: Summarise a production incident as Impact, Cause, and Fix.
---
Return three lines: Impact, Cause, Fix. Use only supplied facts."""
release_skill = """---
name: release-notes
description: Turn a list of merged changes into short user-facing release notes.
---
Write one bullet per user-visible change. Skip internal refactors."""

skills = {}
for text in [adr_skill, incident_skill, release_skill]:
    fields, body = read_skill(text)
    skills[fields["name"]] = {"description": fields["description"], "body": body}
catalog = "\n".join(f"- {name}: {skill['description']}" for name, skill in skills.items())
print(catalog)

## Step 3 · Let the model pick a skill

The prompt holds the catalog and the task, but no skill instructions yet. Our code checks that the reply names a real skill before loading it.

In [ ]:
task = "Our 3-person team ships the course portal as one release. Record the architecture decision."
prompt = f"Available skills:\n{catalog}\n\nTask: {task}\nReply with the name of the one skill to use."
choice = ask(prompt, example="architecture-decision").strip()
if choice not in skills:
    raise ValueError(f"The model picked an unknown skill: {choice!r}")
print("Chosen skill:", choice)
print("Instructions loaded now:\n" + skills[choice]["body"])

## Step 4 · Write a checker for the skill's output

The skill promises three labelled lines. `adr_problems` returns a list of everything that is wrong, and an empty list means the ADR passes. Collecting every problem, instead of stopping at the first one, makes a failing reply easier to fix.

In [ ]:
def adr_problems(adr, expected):
    """List what is wrong with an ADR. An empty list means it passes."""
    pairs = [line.split(":", 1) for line in adr.strip().splitlines() if ":" in line]
    adr_fields = {label.strip(): value.strip() for label, value in pairs}
    problems = []
    if list(adr_fields) != ["Context", "Decision", "Consequence"]:
        problems.append(f"expected Context, Decision, Consequence; got {list(adr_fields)}")
    if adr_fields.get("Decision", "").rstrip(".").lower() != expected:
        problems.append(f"Decision should be {expected!r}")
    return problems

print(adr_problems("Decision: microservices", "modular monolith"))

## Step 5 · Run the skill on two projects

The same skill instructions work for any case. That is the point of packaging them. Each ADR is checked for its format and for its decision.

In [ ]:
cases = [({"team_size": 3, "independent_deploys": False}, "modular monolith"),
         ({"team_size": 12, "independent_deploys": True}, "microservices")]
scripted = ["Context: 3 people; no independent deploys.\nDecision: modular monolith\n"
            "Consequence: One release keeps operations simple, but module boundaries need care.",
            "Context: 12 people; services deploy independently.\nDecision: microservices\n"
            "Consequence: Teams can ship on their own schedules, but operations cost more."]
for (case, expected), example in zip(cases, scripted):
    adr = ask(f"{skills[choice]['body']}\n\nCase: {case}", example=example)
    problems = adr_problems(adr, expected)
    print(f"{adr}\n-> {problems or 'passes'}\n")
    assert not problems, problems

## What you should see

Step 2 prints a three-line catalog. Step 3 prints `Chosen skill: architecture-decision`, followed by its instructions. Step 4 prints two problems for the one-line test ADR. Step 5 prints two ADRs, each followed by `-> passes`.

## Try it

1. Change the task in Step 3 to "Checkout failed for an hour after a bad config push. Write it up." Which skill should be chosen? Set the scripted `example` to that name and run Step 3 only.
2. Add a line such as `\nRisk: Shared code can slow every release.` to the end of the first scripted ADR. Step 5 names the extra label and stops.
3. Write a fourth skill and add it to the loop in Step 2. Does the catalog stay short?

## Takeaway

A skill packages a procedure with a name and a description, so an agent can find it and reuse it. The agent loads only the skill it needs, and your code still checks what the skill produces. Lab 06 hands the review to a separate agent.

## Read further

- [Agent Skills specification](https://agentskills.io/specification)